In [4]:
import numpy as np
import pyscf.gto
import pyscf.scf
import pyscf.cc
import pyscf.fci

from xquces import (
    MolecularHamiltonianLinearOperator,
    hartree_fock_state,
    ucj_from_t_amplitudes,
    ucj_seed_parameters,
    optimize_ucj,
)

bond_length = 0.74
mol = pyscf.gto.M(
    atom=f"H 0 0 0; H 0 0 {bond_length}",
    basis="sto-3g",
    unit="Angstrom",
    spin=0,
    charge=0,
    verbose=0,
)

mf = pyscf.scf.RHF(mol)
hf_energy = mf.kernel()

cc = pyscf.cc.CCSD(mf)
_, t1, t2 = cc.kernel()
cc_energy = cc.e_tot

fci_solver = pyscf.fci.FCI(mf)
fci_energy, _ = fci_solver.kernel()

norb = mf.mo_coeff.shape[1]
nelec = (mol.nelectron // 2, mol.nelectron // 2)
nocc = nelec[0]

hf_vec = hartree_fock_state(norb, nelec)
hop = MolecularHamiltonianLinearOperator.from_scf(mf, nelec=nelec)

seed = ucj_from_t_amplitudes(
    t2,
    t1=t1,
    n_layers=2,
    pair_scale=1.0,
    ov_scale=1.0,
    ov_kick=0.15,
)
seed_vec = seed.apply(hf_vec, nelec=nelec)
seed_energy = hop.expectation(seed_vec)

print("HF   =", hf_energy)
print("Seed =", seed_energy)
print("CCSD =", cc_energy)
print("FCI  =", fci_energy)

x0 = ucj_seed_parameters(
    t2,
    t1=t1,
    n_layers=2,
    pair_scale=1.0,
    ov_scale=1.0,
    ov_kick=1e-3,
)

ansatz_opt, vec_opt, e_opt, res = optimize_ucj(
    x0,
    hamiltonian=hop,
    reference_vec=hf_vec,
    nocc=nocc,
    n_layers=2,
    method="Powell",
    options={"maxiter": 300, "disp": True},
)

print("Optimized UCJ =", e_opt)
print("success       =", res.success)
print("message       =", res.message)
print("x             =", res.x)

HF   = -1.1167593073964255
Seed = -1.0726010442374498
CCSD = -1.1372839986104382
FCI  = -1.1372838344885023
Optimization terminated successfully.
         Current function value: -1.137284
         Iterations: 4
         Function evaluations: 426
Optimized UCJ = -1.137283780709335
success       = True
message       = Optimization terminated successfully.
x             = [ 1.05852426e+00 -2.16029929e+00  1.07912918e+00  2.41556684e-01
 -9.60112925e-04  7.29365994e-02 -2.09196861e-02  1.61054943e-03
  1.30028221e-01  5.22195572e-01]
